# Week 10 Lab — SOLUTIONS — Panel Data Estimators

**MANG2074 Financial Econometrics 1**

**Objectives**

- Estimate pooled OLS, fixed effects and random effects with `linearmodels`.
- Verify the within-transformation/LSDV equivalence numerically.
- Conduct the Hausman test and choose between FE and RE.
- Apply entity-clustered standard errors.

**Data**

- `../data/panelx.csv` — `firm_ident`, `return`, `beta`, `year`: 1,734 firms, 1996–2006.
- `../data/panel_wage.csv` — Cornwell–Rupert wage panel: 595 workers (`id`) × 7 years (`t`); `lwage`, `exp`, `exp2`, `wks`, `ms`, `union`, etc.


## Setup

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels import PooledOLS, PanelOLS, RandomEffects
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# --- firm panel ---
firms = pd.read_csv('../data/panelx.csv')
firms = firms.rename(columns={'return': 'ret'})   # 'return' is a Python keyword
firms = firms.dropna()
firms = firms.set_index(['firm_ident', 'year'])   # MultiIndex: entity, time
print(firms.head())

# --- wage panel ---
wage = pd.read_csv('../data/panel_wage.csv')
wage = wage.set_index(['id', 't'])
print(wage[['lwage', 'exp', 'exp2', 'wks', 'ms', 'union']].describe().round(3))

                      ret      beta
firm_ident year                    
1          1996 -0.004813  1.501050
3          1996 -0.004813  1.116419
4          1996 -0.004813  1.128749
5          1996 -0.001599  1.244929
6          1996 -0.004813  1.611615
          lwage       exp      exp2       wks        ms     union
count  4165.000  4165.000  4165.000  4165.000  4165.000  4165.000
mean      6.676    19.854   514.405    46.812     0.814     0.364
std       0.462    10.966   496.996     5.129     0.389     0.481
min       4.605     1.000     1.000     5.000     0.000     0.000
25%       6.395    11.000   121.000    46.000     1.000     0.000
50%       6.685    18.000   324.000    48.000     1.000     0.000
75%       6.953    29.000   841.000    50.000     1.000     1.000
max       8.537    51.000  2601.000    52.000     1.000     1.000


## Task 1 — Pooled OLS on the firm panel

In [2]:
pooled = PooledOLS.from_formula('ret ~ 1 + beta', firms).fit()
print(pooled)


                          PooledOLS Estimation Summary                          
Dep. Variable:                    ret   R-squared:                     3.118e-06
Estimator:                  PooledOLS   R-squared (Between):             -0.0087
No. Observations:                8856   R-squared (Within):           -9.129e-05
Date:                Thu, Jun 11 2026   R-squared (Overall):           3.118e-06
Time:                        01:30:24   Log-likelihood                 1.357e+04
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      0.0276
Entities:                        1734   P-value                           0.8680
Avg Obs:                       5.1073   Distribution:                  F(1,8854)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             0.0276
                            

**What to interpret.** The pooled slope on beta is ≈ 0.0005 with a p-value of ~0.87: across all firm-years thrown into one big regression, beta earns **no premium whatsoever** — a famously embarrassing result for the basic CAPM (cf. Fama–French 1992). Pooling assumes every firm shares one intercept, i.e. there is *no* unobserved firm-level heterogeneity in average returns; if good firms differ permanently from bad ones, this regression is misspecified.

## Task 2 — Fixed effects

In [3]:
fe = PanelOLS.from_formula('ret ~ 1 + beta + EntityEffects', firms).fit()
print(fe)


                          PanelOLS Estimation Summary                           
Dep. Variable:                    ret   R-squared:                        0.0012
Estimator:                   PanelOLS   R-squared (Between):             -0.0119
No. Observations:                8856   R-squared (Within):               0.0012
Date:                Thu, Jun 11 2026   R-squared (Overall):             -0.0023
Time:                        01:30:24   Log-likelihood                  1.48e+04
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      8.3576
Entities:                        1734   P-value                           0.0039
Avg Obs:                       5.1073   Distribution:                  F(1,7121)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             8.3576
                            

**What to interpret.** With firm fixed effects the slope becomes ≈ **−0.012 and statistically significant** (p ≈ 0.004): *within* a given firm, years when its beta is higher are years when its return is *lower*. The pooled estimate mixed two sources of variation — across firms (between) and over time within firms — and the FE estimator isolates the within part, purged of any time-invariant firm characteristics. A negative within-firm beta–return relation echoes the empirical 'beta anomaly' literature; it is certainly not the upward-sloping SML the textbook CAPM promises.

## Task 3 — FE is just demeaning: verify

In [4]:
ret_dm = firms['ret'] - firms.groupby(level=0)['ret'].transform('mean')
beta_dm = firms['beta'] - firms.groupby(level=0)['beta'].transform('mean')

within = sm.OLS(ret_dm, beta_dm).fit()
print(f"within-OLS slope (by hand) = {within.params.iloc[0]:.6f}")
print(f"PanelOLS FE slope          = {fe.params['beta']:.6f}")


within-OLS slope (by hand) = -0.011893
PanelOLS FE slope          = -0.011893


**What to interpret.** Identical to machine precision. The fixed-effects estimator *is* OLS on entity-demeaned data — equivalently, LSDV with 1,734 firm dummies would give the same slope (we demean rather than invert a 1,735-column design matrix). This is why FE cannot estimate anything constant within a firm: demeaning a constant gives zero. (Footnote: the hand-rolled SEs here are slightly too small because they don't deduct the N estimated means from the degrees of freedom — `PanelOLS` does that correctly.)

## Task 4 — Random effects and the Hausman test

In [5]:
re = RandomEffects.from_formula('ret ~ 1 + beta', firms).fit()
print(re.params.round(4))

b_fe, b_re = fe.params['beta'], re.params['beta']
v_fe, v_re = fe.cov.loc['beta', 'beta'], re.cov.loc['beta', 'beta']

H = (b_fe - b_re)**2 / (v_fe - v_re)
p = stats.chi2.sf(H, df=1)
print(f"\nbeta: FE = {b_fe:.4f}, RE = {b_re:.4f}")
print(f"Hausman H = {H:.3f}, chi2(1) p-value = {p:.4f}")


Intercept    0.0063
beta        -0.0054
Name: parameter, dtype: float64

beta: FE = -0.0119, RE = -0.0054
Hausman H = 6.464, chi2(1) p-value = 0.0110


**What to interpret.** $H_0$: the firm effects are uncorrelated with beta, so RE is consistent (and efficient). $H \approx 6.5$, p ≈ 0.011 — **reject at 5%**, so the RE orthogonality assumption fails and the consistent choice is **fixed effects**. Note the direction: rejecting Hausman points to FE. Economically: firms with persistently high average returns also have systematically different betas, contaminating the RE estimate (which is dragged toward the inconsistent pooled value).

## Task 5 — Cluster-robust standard errors

In [6]:
fe_cl = PanelOLS.from_formula('ret ~ 1 + beta + EntityEffects', firms)\
                .fit(cov_type='clustered', cluster_entity=True)

print(pd.DataFrame({'coef': [fe.params['beta'], fe_cl.params['beta']],
                    'SE': [fe.std_errors['beta'], fe_cl.std_errors['beta']],
                    't': [fe.tstats['beta'], fe_cl.tstats['beta']]},
                   index=['FE, default SE', 'FE, clustered by firm']).round(4))


                         coef      SE       t
FE, default SE        -0.0119  0.0041 -2.8909
FE, clustered by firm -0.0119  0.0053 -2.2363


**What to interpret.** Clustering by firm raises the SE by roughly 30% (0.0041 → 0.0053) and the t-statistic drops accordingly, though significance survives here. Default SEs assume every firm-year is an independent observation; in reality each firm's shocks are correlated across its years, so the panel contains fewer "effective" observations than $NT$. Petersen (2009) showed much published finance research understated uncertainty exactly this way — entity-clustered SEs are now the norm.

## Task 6 — A wage equation

In [7]:
fw = 'lwage ~ 1 + exp + exp2 + wks + ms + union'

pw = PooledOLS.from_formula(fw, wage).fit()
few = PanelOLS.from_formula(fw + ' + EntityEffects', wage).fit()
rew = RandomEffects.from_formula(fw, wage).fit()

table = pd.DataFrame({'pooled OLS': pw.params, 'fixed effects': few.params,
                      'random effects': rew.params})
print(table.round(4))


           pooled OLS  fixed effects  random effects
Intercept      5.8950         4.6156          5.2552
exp            0.0378         0.1136          0.0902
exp2          -0.0007        -0.0004         -0.0008
wks            0.0033         0.0008          0.0009
ms             0.2973        -0.0322         -0.0269
union         -0.0287         0.0301          0.0172


**What to interpret.** The differences are dramatic. The experience profile steepens (0.038 → 0.114 per year within-worker); **marital status flips sign** (+0.30 pooled — the famous 'marriage premium' — to ≈ −0.03 under FE) and **union flips the other way** (−0.03 to +0.03). Interpretation: high-ability workers are *more likely to be married* and *less likely to be unionised*, so pooled OLS credits marriage with what is really ability. FE compares each worker only with himself, sweeping ability out. RE lands between the two — closer to FE but still contaminated. This is the classic demonstration of why panels are so valuable in labour and corporate-finance settings alike.

## Task 7 — Hausman test for the wage equation

In [8]:
slopes = ['exp', 'exp2', 'wks', 'ms', 'union']
b_diff = few.params[slopes] - rew.params[slopes]
V_diff = few.cov.loc[slopes, slopes] - rew.cov.loc[slopes, slopes]

H = float(b_diff.T @ np.linalg.inv(V_diff.values) @ b_diff)
p = stats.chi2.sf(H, df=len(slopes))
print(f"Hausman H = {H:.2f}, chi2({len(slopes)}) p-value = {p:.3g}")


Hausman H = 8838.34, chi2(5) p-value = 0


**What to interpret.** $H$ is in the thousands with a p-value of zero: the RE assumption is annihilated — unobserved worker ability is massively correlated with the regressors. **Fixed effects is the only defensible estimator** for this wage equation. (When $V_{FE}-V_{RE}$ is not positive definite in small samples, the test can fail mechanically; here it is comfortably well-behaved.)

## Task 8 — Write-up

**Beta and returns.** Pooled OLS finds no cross-sectional reward for beta; controlling for firm fixed effects, the within-firm relation is significantly *negative* (≈ −0.012 per unit of beta, clustered t ≈ −2.3). The Hausman test (p ≈ 0.01) rejects random effects, so FE with firm-clustered SEs is our preferred specification. This is evidence against a naive SML and consistent with the low-beta anomaly — though omitted time effects (market-wide years) and measurement error in estimated betas deserve attention before strong conclusions.

**Wages.** Pooled estimates are badly contaminated by worker heterogeneity: the marriage premium vanishes (indeed flips) and the union effect turns positive once fixed effects absorb ability. The Hausman test rejects RE overwhelmingly, so we report FE. General lesson: in panels where entities plausibly differ in unobserved, persistent ways correlated with the regressors — i.e. almost always in finance — FE plus clustered SEs is the safe default.